In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from data_loader import (
    load_all_splits, verify_data,
    CLASSES, SORTED_CLASSES, DATA_DIR,
)

print('Classes (alphabetical, index = label):')
for i, c in enumerate(SORTED_CLASSES):
    print(f'  {i} = {c}')

Classes (alphabetical, index = label):
  0 = construction_site
  1 = debris_or_rubble
  2 = fire_station
  3 = flooded_road
  4 = hospital
  5 = impoverished_settlement
  6 = multi-unit_residential
  7 = single-unit_residential


In [2]:
verify_data()

Data check: C:\Users\rdlou\class\data402\finalproject\data

processed:
  construction_site            3326
  debris_or_rubble             2098
  fire_station                 1348
  flooded_road                 2073
  hospital                     1019
  impoverished_settlement      1366
  multi-unit_residential       1373
  single-unit_residential      1500
  total                       14103

processed_val:
  construction_site              40
  debris_or_rubble               18
  fire_station                   94
  flooded_road                   40
  hospital                       73
  impoverished_settlement        16
  multi-unit_residential         99
  single-unit_residential       200
  total                         580

processed_test:
  construction_site              40
  debris_or_rubble               19
  fire_station                   95
  flooded_road                   40
  hospital                       74
  impoverished_settlement        17
  multi-unit_residential        

In [3]:
X_train, y_train, X_val, y_val, X_test, y_test = load_all_splits(size=64)

Training set:
  construction_site            3326
  debris_or_rubble             2098
  fire_station                 1348
  flooded_road                 2073
  hospital                     1019
  impoverished_settlement      1366
  multi-unit_residential       1373
  single-unit_residential      1500
Validation set:
  construction_site              40
  debris_or_rubble               18
  fire_station                   94
  flooded_road                   40
  hospital                       73
  impoverished_settlement        16
  multi-unit_residential         99
  single-unit_residential       200
Test set:
  construction_site              40
  debris_or_rubble               19
  fire_station                   95
  flooded_road                   40
  hospital                       74
  impoverished_settlement        17
  multi-unit_residential        100
  single-unit_residential       200

Shapes: train (14103, 12288), val (580, 12288), test (585, 12288)


In [4]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train.astype(np.float32))
X_val_s   = scaler.transform(X_val.astype(np.float32))
X_test_s  = scaler.transform(X_test.astype(np.float32))

print(f'X_train_s: {X_train_s.shape}, dtype={X_train_s.dtype}')

X_train_s: (14103, 12288), dtype=float32


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

best_f1, best_C, best_clf = -1, None, None

for C in [0.001, 0.01, 0.1, 1.0, 10.0]: # Doing this to see how regularized we should make the model (lower C = more simple / stronger regularization, larger C = opposite)
    clf = LogisticRegression(
        max_iter=1000,
        class_weight='balanced',
        solver='lbfgs',
        C=C,
        n_jobs=-1,
    )
    clf.fit(X_train_s, y_train)
    val_pred = clf.predict(X_val_s)
    acc = (val_pred == y_val).mean()
    f1 = f1_score(y_val, val_pred, average='macro')
    print(f'C={C:<7}  val acc: {acc:.3f}   macro-F1: {f1:.3f}')
    if f1 > best_f1:
        best_f1, best_C, best_clf = f1, C, clf

print(f'\nBest: C={best_C}, val macro-F1={best_f1:.3f}')

| C       | Validation Accuracy | Macro-F1 |
|---------|---------------------|----------|
| 0.001   | 0.234               | 0.203    |
| 0.01    | 0.228               | 0.188    |
| 0.1     | 0.212               | 0.175    |
| 1.0     | 0.184               | 0.153    |
| 10.0    | 0.193               | 0.161    |

**Best:** `C = 0.001`, validation macro-F1 = **0.203**

The sweep selects the strongest regularization in our grid, which makes sense given the model has ~12,000 features (64×64×3 pixels) and only ~14,000 training examples — without aggressive regularization, the linear model overfits to pixel-level noise. The fact that the best C sits at the edge of the grid suggests trying even smaller values (e.g., 0.0001, 0.00001) could squeeze out a small additional gain

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

val_pred = best_clf.predict(X_val_s)

print('Validation classification report:')
print(classification_report(y_val, val_pred,
                            target_names=SORTED_CLASSES, digits=3))

print('\nValidation confusion matrix:')
cm = confusion_matrix(y_val, val_pred)
print(pd.DataFrame(cm, index=SORTED_CLASSES, columns=SORTED_CLASSES))

### Validation Classification Report

| Class                   | Precision | Recall | F1-Score | Support |
|-------------------------|-----------|--------|----------|---------|
| construction_site       | 0.111     | 0.175  | 0.136    | 40      |
| debris_or_rubble        | 0.062     | 0.222  | 0.096    | 18      |
| fire_station            | 0.304     | 0.298  | 0.301    | 94      |
| flooded_road            | 0.234     | 0.450  | 0.308    | 40      |
| hospital                | 0.259     | 0.192  | 0.220    | 73      |
| impoverished_settlement | 0.045     | 0.125  | 0.067    | 16      |
| multi-unit_residential  | 0.216     | 0.192  | 0.203    | 99      |
| single-unit_residential | 0.454     | 0.220  | 0.296    | 200     |
| **accuracy**            |           |        | **0.234**| **580** |
| **macro avg**           | 0.211     | 0.234  | 0.203    | 580     |
| **weighted avg**        | 0.302     | 0.234  | 0.249    | 580     |

### Validation Confusion Matrix

Rows are true class, columns are predicted class. Abbreviations: CS = construction_site, DR = debris_or_rubble, FS = fire_station, FR = flooded_road, HS = hospital, IS = impoverished_settlement, MR = multi-unit_residential, SR = single-unit_residential.

| True \ Pred | CS  | DR  | FS  | FR  | HS  | IS  | MR  | SR  |
|-------------|-----|-----|-----|-----|-----|-----|-----|-----|
| **CS**      | 7   | 7   | 6   | 4   | 2   | 5   | 5   | 4   |
| **DR**      | 1   | 4   | 1   | 2   | 2   | 1   | 3   | 4   |
| **FS**      | 10  | 4   | 28  | 13  | 7   | 4   | 13  | 15  |
| **FR**      | 5   | 6   | 4   | 18  | 2   | 1   | 2   | 2   |
| **HS**      | 10  | 9   | 4   | 11  | 14  | 6   | 12  | 7   |
| **IS**      | 3   | 2   | 2   | 1   | 1   | 2   | 5   | 0   |
| **MR**      | 11  | 9   | 18  | 5   | 10  | 6   | 19  | 21  |
| **SR**      | 16  | 24  | 29  | 23  | 16  | 19  | 29  | 44  |

### Discussion

The logistic regression baseline achieves a validation macro-F1 of 0.203 — meaningfully above the random-guessing floor (~0.125 for 8 balanced classes) but well below useful performance. Several patterns are worth highlighting:

The two best-performing classes are **flooded_road** (F1 = 0.308, recall 45%) and **fire_station** (F1 = 0.301). The flooded_road result is somewhat surprising given the dataset's known label-noise issue, where multiple temporal views of the same location can show the road both dry and flooded — the linear model still picks up enough water-pixel signal to do better than average. fire_station's structure (often standalone buildings with distinctive parking-lot adjacencies) appears to be linearly separable enough to get reasonable signal from raw pixels.

The worst-performing classes are **impoverished_settlement** (F1 = 0.067) and **debris_or_rubble** (F1 = 0.096), both starved by extremely small validation sets (16 and 18 examples respectively) and by the linear model's inability to capture the spatial texture cues that distinguish these classes from residential areas and construction sites.

The confusion matrix reveals the model's core weakness: **single-unit_residential** has 200 validation examples but only 44 are correctly identified — its predictions are scattered fairly evenly across all other classes, suggesting the model has no coherent decision boundary for it in raw-pixel space. The expected confusion between **single-unit and multi-unit residential** (21 SR predicted as MR, 29 MR predicted as SR) is visible, as is the **debris vs construction_site** confusion that motivated keeping construction_site as a confounder class in our sampling.

These results establish the floor we'd expect from a model that lacks any notion of spatial structure. The CNN and pretrained ResNet should close most of this gap by learning local visual features that no linear model can capture from flattened pixel intensities.

## Convolution Neural Network

In [9]:
import torch
import torchvision
from torchvision import transforms
from torchvision.datasets import ImageFolder
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
NVIDIA GeForce RTX 3060 Laptop GPU


### Loading images in as tensors and doing basic transformations to help the CNN

In [13]:
transforms_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(degrees=180),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

transforms_val = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

train_ds = ImageFolder('data/processed',      transform=transforms_train)
val_ds   = ImageFolder('data/processed_val',  transform=transforms_val)
test_ds  = ImageFolder('data/processed_test', transform=transforms_val)

print(len(train_ds))
print(train_ds.classes)
img, label = train_ds[0]
print(img.shape, img.dtype)
print(label)

14103
['construction_site', 'debris_or_rubble', 'fire_station', 'flooded_road', 'hospital', 'impoverished_settlement', 'multi-unit_residential', 'single-unit_residential']
torch.Size([3, 224, 224]) torch.float32
0


### Batching the data

In [18]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=0)

images, labels = next(iter(train_loader))
print(images.shape)
print(labels.shape)
print(labels[:10])

torch.Size([32, 3, 224, 224])
torch.Size([32])
tensor([2, 0, 0, 0, 1, 5, 3, 5, 6, 0])


### Making the CNN from scratch class and doing quick test to make sure pipeline works

In [19]:
import torch.nn as nn

class FmowCNN(nn.Module):
    def __init__(self, num_classes=8):
        super().__init__()
        
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        
        self.conv4 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.bn4 = nn.BatchNorm2d(256)
        
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(kernel_size=2)
        
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(256, 8)
    
    def forward(self, x):
        x = self.pool(self.relu(self.bn1(self.conv1(x))))
        x = self.pool(self.relu(self.bn2(self.conv2(x))))
        x = self.pool(self.relu(self.bn3(self.conv3(x))))
        x = self.pool(self.relu(self.bn4(self.conv4(x))))
        x = self.global_pool(x)
        x = x.view(x.size(0), -1)
        x = self.dropout(x)
        x = self.fc(x)
        return x

model = FmowCNN(num_classes=8)
print(model)
dummy = torch.randn(2, 3, 224, 224)
output = model(dummy)
print(output.shape)

FmowCNN(
  (conv1): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv3): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn3): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv4): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn4): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU()
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (global_pool): AdaptiveAvgPool2d(output_size=1)
  (dropout): Dropout(p=0.5, inplace=False)
  (fc): Linear(in_features=256, out_features=8, bias=True)
)
torch.Size([2, 8])


### Setting to gpu for speed

In [20]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


### Adding class weights

In [21]:
import numpy as np
train_labels = [label for _, label in train_ds.samples]
class_counts = np.bincount(train_labels)
class_weights = len(train_labels) / (len(class_counts) * class_counts)
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)
print('Class weights:', class_weights)

Class weights: tensor([0.5300, 0.8403, 1.3078, 0.8504, 1.7300, 1.2905, 1.2840, 1.1753],
       device='cuda:0')


### Making model instance and loss function + optimizer

In [22]:
model = FmowCNN(num_classes=8).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [26]:
import time
NUM_EPOCHS = 30

best_val_f1 = -1

for epoch in range(NUM_EPOCHS):
    model.train()
    train_loss_sum = 0
    train_correct = 0
    train_total = 0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        
        train_loss_sum += loss.item() * images.size(0)
        train_correct += (logits.argmax(1) == labels).sum().item()
        train_total += labels.size(0)
    
    train_loss = train_loss_sum / train_total
    train_acc = train_correct / train_total
    
    model.eval()
    val_loss_sum = 0
    val_correct = 0
    val_total = 0
    all_preds, all_labels = [], []
    
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            logits = model(images)
            loss = criterion(logits, labels)
            
            val_loss_sum += loss.item() *images.size(0)
            preds = logits.argmax(1)
            val_correct += (preds == labels).sum().item()
            val_total += images.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
        
        val_loss = val_loss_sum / val_total
        val_acc = val_correct / val_total
        
        from sklearn.metrics import f1_score
        val_f1 = f1_score(all_labels, all_preds, average='macro')
        
        print(f'Epoch {epoch+1:2d}/{NUM_EPOCHS} | '
          f'train loss {train_loss:.3f} acc {train_acc:.3f} | '
          f'val loss {val_loss:.3f} acc {val_acc:.3f} f1 {val_f1:.3f}')
        
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), 'best_cnn.pth')
        print(f'  saved new best model (val F1 = {val_f1:.3f})')
        
    time.sleep(5)


Epoch  1/30 | train loss 1.353 acc 0.495 | val loss 1.436 acc 0.433 f1 0.420
  saved new best model (val F1 = 0.420)
Epoch  2/30 | train loss 1.339 acc 0.502 | val loss 1.493 acc 0.464 f1 0.445
  saved new best model (val F1 = 0.445)
Epoch  3/30 | train loss 1.331 acc 0.500 | val loss 1.395 acc 0.472 f1 0.454
  saved new best model (val F1 = 0.454)
Epoch  4/30 | train loss 1.326 acc 0.505 | val loss 1.339 acc 0.500 f1 0.469
  saved new best model (val F1 = 0.469)
Epoch  5/30 | train loss 1.321 acc 0.507 | val loss 1.326 acc 0.505 f1 0.474
  saved new best model (val F1 = 0.474)
Epoch  6/30 | train loss 1.302 acc 0.515 | val loss 1.304 acc 0.493 f1 0.460
Epoch  7/30 | train loss 1.300 acc 0.521 | val loss 1.394 acc 0.481 f1 0.456
Epoch  8/30 | train loss 1.296 acc 0.516 | val loss 1.325 acc 0.517 f1 0.461
Epoch  9/30 | train loss 1.278 acc 0.526 | val loss 1.311 acc 0.503 f1 0.487
  saved new best model (val F1 = 0.487)
Epoch 10/30 | train loss 1.274 acc 0.523 | val loss 1.323 acc 0.519

In [27]:
model.load_state_dict(torch.load('best_cnn.pth'))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        preds = model(images).argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

from sklearn.metrics import classification_report, confusion_matrix
print(classification_report(all_labels, all_preds, target_names=SORTED_CLASSES, digits=3))

C:\Users\rdlou\AppData\Local\Temp\ipykernel_9748\1003283002.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('best_cnn.pth'))


                         precision    recall  f1-score   support

      construction_site      0.476     0.500     0.488        40
       debris_or_rubble      0.238     0.556     0.333        18
           fire_station      0.513     0.628     0.565        94
           flooded_road      0.515     0.875     0.648        40
               hospital      0.704     0.260     0.380        73
impoverished_settlement      0.875     0.875     0.875        16
 multi-unit_residential      0.355     0.273     0.309        99
single-unit_residential      0.701     0.680     0.690       200

               accuracy                          0.552       580
              macro avg      0.547     0.581     0.536       580
           weighted avg      0.574     0.552     0.543       580



### From-Scratch CNN — Training Log

| Epoch | Train Loss | Train Acc | Val Loss | Val Acc | Val F1 | Saved |
|-------|------------|-----------|----------|---------|--------|-------|
| 1     | 1.353      | 0.495     | 1.436    | 0.433   | 0.420  | ✓     |
| 2     | 1.339      | 0.502     | 1.493    | 0.464   | 0.445  | ✓     |
| 3     | 1.331      | 0.500     | 1.395    | 0.472   | 0.454  | ✓     |
| 4     | 1.326      | 0.505     | 1.339    | 0.500   | 0.469  | ✓     |
| 5     | 1.321      | 0.507     | 1.326    | 0.505   | 0.474  | ✓     |
| 6     | 1.302      | 0.515     | 1.304    | 0.493   | 0.460  |       |
| 7     | 1.300      | 0.521     | 1.394    | 0.481   | 0.456  |       |
| 8     | 1.296      | 0.516     | 1.325    | 0.517   | 0.461  |       |
| 9     | 1.278      | 0.526     | 1.311    | 0.503   | 0.487  | ✓     |
| 10    | 1.274      | 0.523     | 1.323    | 0.519   | 0.496  | ✓     |
| 11    | 1.263      | 0.529     | 1.323    | 0.500   | 0.506  | ✓     |
| 12    | 1.261      | 0.529     | 1.313    | 0.512   | 0.504  |       |
| 13    | 1.254      | 0.531     | 1.309    | 0.516   | 0.506  |       |
| 14    | 1.244      | 0.535     | 1.346    | 0.509   | 0.506  |       |
| 15    | 1.250      | 0.537     | 1.275    | 0.529   | 0.495  |       |
| 16    | 1.232      | 0.539     | 1.279    | 0.534   | 0.499  |       |
| 17    | 1.228      | 0.542     | 1.400    | 0.486   | 0.476  |       |
| 18    | 1.221      | 0.543     | 1.256    | 0.550   | 0.522  | ✓     |
| 19    | 1.219      | 0.545     | 1.297    | 0.524   | 0.502  |       |
| 20    | 1.215      | 0.552     | 1.378    | 0.503   | 0.473  |       |
| 21    | 1.199      | 0.546     | 1.360    | 0.478   | 0.473  |       |
| 22    | 1.200      | 0.556     | 1.252    | 0.541   | 0.527  | ✓     |
| 23    | 1.196      | 0.554     | 1.262    | 0.548   | 0.535  | ✓     |
| 24    | 1.194      | 0.559     | 1.292    | 0.531   | 0.514  |       |
| 25    | 1.188      | 0.554     | 1.233    | 0.552   | **0.536** | ✓ |
| 26    | 1.181      | 0.563     | 1.200    | 0.548   | 0.529  |       |
| 27    | 1.178      | 0.557     | 1.462    | 0.457   | 0.436  |       |
| 28    | 1.180      | 0.560     | 1.344    | 0.503   | 0.502  |       |
| 29    | 1.175      | 0.562     | 1.334    | 0.545   | 0.532  |       |
| 30    | 1.169      | 0.567     | 1.369    | 0.509   | 0.478  |       |

**Best model saved at epoch 25** with validation macro-F1 = **0.536**.
Train accuracy and val accuracy track each other closely throughout (e.g. epoch 25: 0.554 vs 0.552), indicating no significant overfitting.

### From-Scratch CNN — Validation Classification Report (best model)

| Class                   | Precision | Recall | F1-Score | Support |
|-------------------------|-----------|--------|----------|---------|
| construction_site       | 0.476     | 0.500  | 0.488    | 40      |
| debris_or_rubble        | 0.238     | 0.556  | 0.333    | 18      |
| fire_station            | 0.513     | 0.628  | 0.565    | 94      |
| flooded_road            | 0.515     | 0.875  | 0.648    | 40      |
| hospital                | 0.704     | 0.260  | 0.380    | 73      |
| impoverished_settlement | 0.875     | 0.875  | 0.875    | 16      |
| multi-unit_residential  | 0.355     | 0.273  | 0.309    | 99      |
| single-unit_residential | 0.701     | 0.680  | 0.690    | 200     |
| **accuracy**            |           |        | **0.552**| **580** |
| **macro avg**           | 0.547     | 0.581  | 0.536    | 580     |
| **weighted avg**        | 0.574     | 0.552  | 0.543    | 580     |

### Discussion

The from-scratch CNN achieves validation macro-F1 = 0.536, a 2.6× improvement over the logistic regression baseline (0.203). The largest per-class gains come from classes with distinctive spatial structure:

- **impoverished_settlement** F1: 0.067 → 0.875 — dense, irregular layouts are spatially distinctive
- **single-unit_residential** F1: 0.296 → 0.690 — regular spacing, yards, driveways
- **flooded_road** F1: 0.308 → 0.648 — recall 0.875, the model finds nearly every flooded road
- **fire_station** F1: 0.301 → 0.565 — distinctive building footprint

The remaining failure modes:

- **hospital** (F1 = 0.380): high precision (0.704), low recall (0.260) — when the model predicts hospital it's usually right, but it misses most actual hospitals, likely confusing them with multi-unit_residential or large institutional buildings
- **multi-unit_residential** (F1 = 0.309): still confused with single-unit_residential and other large-footprint classes
- **debris_or_rubble** (F1 = 0.333): high recall (0.556) but low precision (0.238) — the model over-predicts debris, likely including some construction_site examples

Training shows no significant overfitting (train acc 0.554 vs val acc 0.552 at the best epoch), suggesting room for additional capacity or training time. The augmentation strategy (random flips, vertical flips, full-range rotations) and class weighting appear well-calibrated for this dataset.

## ResNet Transfer Learning CNN Model

In [29]:
from torchvision import models

model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

num_features = model.fc.in_features      # whatever the final layer's input size is
model.fc = nn.Linear(num_features, 8)    # replace with our 8-class head

model = model.to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=.0001)

In [30]:
import time
NUM_EPOCHS = 30

best_val_f1 = -1

for epoch in range(NUM_EPOCHS):
    model.train()
    train_loss_sum = 0
    train_correct = 0
    train_total = 0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        
        train_loss_sum += loss.item() * images.size(0)
        train_correct += (logits.argmax(1) == labels).sum().item()
        train_total += labels.size(0)
    
    train_loss = train_loss_sum / train_total
    train_acc = train_correct / train_total
    
    model.eval()
    val_loss_sum = 0
    val_correct = 0
    val_total = 0
    all_preds, all_labels = [], []
    
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            logits = model(images)
            loss = criterion(logits, labels)
            
            val_loss_sum += loss.item() *images.size(0)
            preds = logits.argmax(1)
            val_correct += (preds == labels).sum().item()
            val_total += images.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
        
        val_loss = val_loss_sum / val_total
        val_acc = val_correct / val_total
        
        from sklearn.metrics import f1_score
        val_f1 = f1_score(all_labels, all_preds, average='macro')
        
        print(f'Epoch {epoch+1:2d}/{NUM_EPOCHS} | '
          f'train loss {train_loss:.3f} acc {train_acc:.3f} | '
          f'val loss {val_loss:.3f} acc {val_acc:.3f} f1 {val_f1:.3f}')
        
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), 'best_resnet.pth')
        print(f'  saved new best model (val F1 = {val_f1:.3f})')
        
    time.sleep(5)


Epoch  1/30 | train loss 1.140 acc 0.588 | val loss 1.169 acc 0.579 f1 0.578
  saved new best model (val F1 = 0.578)
Epoch  2/30 | train loss 0.927 acc 0.669 | val loss 1.020 acc 0.629 f1 0.621
  saved new best model (val F1 = 0.621)
Epoch  3/30 | train loss 0.846 acc 0.699 | val loss 1.078 acc 0.597 f1 0.617
Epoch  4/30 | train loss 0.767 acc 0.727 | val loss 1.086 acc 0.621 f1 0.609
Epoch  5/30 | train loss 0.714 acc 0.746 | val loss 1.064 acc 0.610 f1 0.617
Epoch  6/30 | train loss 0.678 acc 0.760 | val loss 1.046 acc 0.633 f1 0.637
  saved new best model (val F1 = 0.637)
Epoch  7/30 | train loss 0.629 acc 0.776 | val loss 0.933 acc 0.667 f1 0.668
  saved new best model (val F1 = 0.668)
Epoch  8/30 | train loss 0.600 acc 0.786 | val loss 1.041 acc 0.662 f1 0.651
Epoch  9/30 | train loss 0.562 acc 0.801 | val loss 0.980 acc 0.641 f1 0.646
Epoch 10/30 | train loss 0.536 acc 0.809 | val loss 1.100 acc 0.614 f1 0.614
Epoch 11/30 | train loss 0.508 acc 0.817 | val loss 1.015 acc 0.640 f1

In [ ]:
model.load_state_dict(torch.load('best_resnet.pth'))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        preds = model(images).argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

from sklearn.metrics import classification_report, confusion_matrix
print(classification_report(all_labels, all_preds, target_names=SORTED_CLASSES, digits=3))

### Pretrained ResNet-18 — Validation Classification Report (best model)

Best epoch: 13. Validation macro-F1 = **0.671**, validation accuracy = **0.666**.

| Class                   | Precision | Recall | F1-Score | Support |
|-------------------------|-----------|--------|----------|---------|
| construction_site       | 0.778     | 0.525  | 0.627    | 40      |
| debris_or_rubble        | 0.379     | 0.611  | 0.468    | 18      |
| fire_station            | 0.626     | 0.819  | 0.710    | 94      |
| flooded_road            | 0.776     | 0.950  | 0.854    | 40      |
| hospital                | 0.705     | 0.589  | 0.642    | 73      |
| impoverished_settlement | 0.833     | 0.938  | 0.882    | 16      |
| multi-unit_residential  | 0.479     | 0.465  | 0.472    | 99      |
| single-unit_residential | 0.763     | 0.675  | 0.716    | 200     |
| **accuracy**            |           |        | **0.666**| **580** |
| **macro avg**           | 0.667     | 0.696  | 0.671    | 580     |
| **weighted avg**        | 0.677     | 0.666  | 0.664    | 580     |

### Per-Class Improvement Across Models

| Class                   | LR F1 | CNN F1 | ResNet F1 | LR → ResNet Gain |
|-------------------------|-------|--------|-----------|------------------|
| construction_site       | 0.136 | 0.488  | 0.627     | +0.491           |
| debris_or_rubble        | 0.096 | 0.333  | 0.468     | +0.372           |
| fire_station            | 0.301 | 0.565  | 0.710     | +0.409           |
| flooded_road            | 0.308 | 0.648  | 0.854     | +0.546           |
| hospital                | 0.220 | 0.380  | 0.642     | +0.422           |
| impoverished_settlement | 0.067 | 0.875  | 0.882     | +0.815           |
| multi-unit_residential  | 0.203 | 0.309  | 0.472     | +0.269           |
| single-unit_residential | 0.296 | 0.690  | 0.716     | +0.420           |

### Discussion

The pretrained ResNet-18 achieves validation macro-F1 = 0.671, a substantial improvement over both the logistic regression baseline (0.203) and the from-scratch CNN (0.536). Every class improved, and the largest absolute gains came from classes that were difficult for both prior models:

- **hospital** F1: 0.380 → 0.642. The from-scratch CNN had high precision (0.704) but very poor recall (0.260) on this class — it recognized hospitals when it was confident but missed most. The pretrained ResNet's transfer learning resolves this by bringing in richer features that can distinguish hospital architecture from generic large buildings, lifting recall to 0.589 while maintaining precision (0.705).

- **flooded_road** F1: 0.648 → 0.854. Recall climbs from 0.875 to 0.950, while precision jumps from 0.515 to 0.776 — the model is now catching almost every flooded road *and* not over-predicting them. For a disaster-response use case this is the most valuable class to improve.

- **multi-unit_residential** F1: 0.309 → 0.472. Still the weakest single-unit class to identify, but materially better. The remaining confusion is likely with hospital (large multi-building footprints) and single-unit_residential (housing).

- **construction_site** F1: 0.488 → 0.627. The "confounder" class is now well-separated from debris_or_rubble.

The model's lowest per-class F1 remains **debris_or_rubble (0.468)**, with high recall (0.611) but low precision (0.379). The model finds debris when it's present but over-predicts it, likely confusing some construction sites and damaged-looking areas. This class also has the smallest validation support (18 examples), so per-class metrics carry meaningful variance.

### Training Behavior — Overfitting

Validation metrics peak at epoch 13 and decline thereafter. By epoch 30, training accuracy reaches 0.935 while validation accuracy holds at 0.628, a 30-point gap, and validation loss has increased from its minimum of 0.933 (epoch 7) to 1.589 (epoch 30) — a clear overfitting signature. This is expected behavior when fine-tuning an 11-million-parameter pretrained model on a modest 14k-image training set: the model has the capacity to memorize training examples faster than it can find generalizable patterns. The save-best-model strategy captured the optimal generalization point at epoch 13, so the deployed model is unaffected by the later overfitting. In retrospect, training could have been stopped at epoch 15 with no loss in final performance, or early stopping could have been added to terminate automatically once validation metrics stopped improving.

### Three-Model Comparison Summary

| Model                  | Val Accuracy | Val Macro-F1 | Notes                                |
|------------------------|--------------|--------------|--------------------------------------|
| Logistic Regression    | 0.234        | 0.203        | Linear, no spatial structure         |
| From-scratch CNN       | 0.552        | 0.536        | Spatial structure, trained on 14k    |
| Pretrained ResNet-18   | **0.666**    | **0.671**    | Spatial structure + ImageNet transfer |

Each successive model adds one specific capability and the metrics quantify the gain. The 2.6× improvement from LR to CNN demonstrates the value of respecting spatial structure in image data. The further 25% improvement from CNN to ResNet demonstrates the value of transfer learning when training data is limited — a pretrained model's prior knowledge from millions of ImageNet images more than compensates for the modest 14k-image fine-tuning set.